# Nanopore analysis (from the PRJEB88111 dataset)

data from [Metagenomics provides broad detection of pathogens, antimicrobial resistance, and virulence genes in pig diarrhoea and complement conventional methods](https://link.springer.com/article/10.1186/s42523-026-00577-2), Ivanova et al., 2026.

SRA entry: https://www.ncbi.nlm.nih.gov/sra/?term=PRJEB88111

In [1]:
import polars as pl
import matplotlib.pyplot as plt

## Read in run & sample metadata

In [2]:
expinfo_df = (
    pl.read_csv('../inputs.PRJEB88111.nanopore/experiment_info.PRJEB88111.csv')
    .filter(pl.col("Organism Name") == 'Sus scrofa')
)
expinfo_df.head(1)

Experiment Accession,Experiment Title,Organism Name,Instrument,Submitter,Study Accession,Study Title,Sample Accession,Sample Title,"Total Size, Mb",Total RUNs,Total Spots,Total Bases,Library Name,Library Strategy,Library Source,Library Selection
str,str,str,str,str,str,str,str,str,f64,i64,i64,i64,str,str,str,str
"""ERX15330493""","""Raw reads: DTU_2024_1036881_10…","""Sus scrofa""","""PromethION""","""Centre for Genomic Epidemiolog…","""ERP171251""","""Metagenomics for detection of …","""ERS27373427""","""""",28855.43,1,9597516,35068056814,"""""","""WGS""","""METAGENOMIC""","""RANDOM"""


In [3]:
# per https://link.springer.com/article/10.1186/s42523-026-00577-2, the LVK datasets are the diarrhea datasets.
xx = []
for row in expinfo_df.iter_rows(named=True):
    exp_acc = row["Experiment Accession"]

    is_lvk = 0
    title = row["Experiment Title"]
    if "LVK" in title:
        is_lvk = 1

    xx.append(dict(is_diarrhea=is_lvk, Experiment=exp_acc))

xx_df = pl.DataFrame(xx)
xx_df

is_diarrhea,Experiment
i64,str
1,"""ERX15330493"""
1,"""ERX15330568"""
1,"""ERX15332128"""
1,"""ERX15333324"""
1,"""ERX15330180"""
…,…
1,"""ERX15305647"""
1,"""ERX15305652"""
1,"""ERX15305662"""


In [4]:
runinfo_df = (
    pl.read_csv('../inputs.PRJEB88111.nanopore/SraRunInfo.PRJEB88111.csv')
    .filter(pl.col("ScientificName") == 'Sus scrofa')
)
runinfo_df.head(1)

Run,ReleaseDate,LoadDate,spots,bases,spots_with_mates,avgLength,size_MB,AssemblyName,download_path,Experiment,LibraryName,LibraryStrategy,LibrarySelection,LibrarySource,LibraryLayout,InsertSize,InsertDev,Platform,Model,SRAStudy,BioProject,Study_Pubmed_id,ProjectID,Sample,BioSample,SampleType,TaxID,ScientificName,SampleName,g1k_pop_code,source,g1k_analysis_group,Subject_ID,Sex,Disease,Tumor,Affection_Status,Analyte_Type,Histological_Type,Body_Site,CenterName,Submission,dbgap_study_accession,Consent,RunHash,ReadHash
str,str,str,i64,i64,i64,i64,i64,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,i64,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""ERR15936676""","""2025-11-21 21:12:07""","""2025-11-25 01:08:24""",5640279,23038698918,0,4084,18891,null,"""https://sra-downloadb.be-md.nc…","""ERX15332274""",null,"""WGS""","""RANDOM""","""METAGENOMIC""","""SINGLE""",0,0,"""OXFORD_NANOPORE""","""PromethION""","""ERP171251""","""PRJEB88111""",null,1257717,"""ERS27373425""","""SAMEA120577422""","""simple""",9823,"""Sus scrofa""","""SAMEA120577422""",null,null,null,null,null,null,"""no""",null,null,null,null,"""CENTRE FOR GENOMIC EPIDEMIOLOG…","""ERA35260797""",null,"""public""","""58DD49D01233D0BB226B8858217137…","""EF19728837C856F4CD0B157146C8E2…"


In [5]:
join_df = (
    runinfo_df
    .join(xx_df, on="Experiment", how="inner")
    .select(["Run", "Model", "is_diarrhea", "bases"])
)
join_df['Model'].value_counts()

Model,count
str,u32
"""GridION""",27
"""PromethION""",32


In [6]:
# select just the PromethION since they are deeper
prjeb_df = join_df.filter(pl.col('Model') == 'PromethION')
prjeb_df.head(1)

Run,Model,is_diarrhea,bases
str,str,i64,i64
"""ERR15934865""","""PromethION""",1,35068056814


## Read in summary of mapping of nanopore to reference genomes

In [7]:
depth_df = (
    pl.read_csv('../outputs.PRJEB88111.nanopore/bams/*.x.all-core.depth.csv')
)
depth_df.head(1)

metag,gene,len,hits,breadth,depth_all,depth_cov
str,str,i64,i64,f64,f64,f64
"""ERR15909852""","""FR882202.1""",72839,5938,0.081522,0.088785,1.089087


## Read in table connecting contigs to species

In [8]:
contigs_species_df = pl.read_csv('../outputs.PRJEB88111.nanopore/contigs_to_species.csv')
contigs_species_df.head(1)

contig,species,acc
str,str,str
"""UREV01000001.1""","""s__Cryptobacteroides sp9005469…","""GCA_900546925.1"""


## Combine depth and contigs to species!!

In [9]:
combo_df = (
    depth_df
    .join(contigs_species_df, left_on='gene', right_on='contig', how='inner')
    .with_columns(
        weighted_depth=(pl.col('len') * pl.col('depth_all')),
        weighted_breadth=(pl.col('len') * pl.col('breadth')),
    )
)
combo_df.head(1)

metag,gene,len,hits,breadth,depth_all,depth_cov,species,acc,weighted_depth,weighted_breadth
str,str,i64,i64,f64,f64,f64,str,str,f64,f64
"""ERR15909852""","""FR882202.1""",72839,5938,0.081522,0.088785,1.089087,"""s__Cryptobacteroides sp0004349…","""GCA_000434935.1""",6467.0,5938.0


## Summarize by metag/species

In [10]:
sum_df = combo_df.group_by(['metag', 'species']).agg(
    depth_all=pl.col('weighted_depth').sum() / pl.col('len').sum(),
    breadth=pl.col('weighted_breadth').sum() / pl.col('len').sum(),
).join(prjeb_df, left_on='metag', right_on='Run', how='inner')
sum_df.sort('breadth')

metag,species,depth_all,breadth,Model,is_diarrhea,bases
str,str,f64,f64,str,i64,i64
"""ERR15934505""","""s__Ornithospirochaeta sp022785…",0.204572,0.056841,"""PromethION""",1,7393986829
"""ERR15934505""","""s__UBA2868 sp004552595""",2.94368,0.192117,"""PromethION""",1,7393986829
"""ERR15934505""","""s__Cryptobacteroides sp9005469…",0.503539,0.300202,"""PromethION""",1,7393986829
"""ERR15939764""","""s__UBA2868 sp004552595""",1.812845,0.328263,"""PromethION""",1,6640569258
"""ERR15934783""","""s__UBA2868 sp004552595""",18.441043,0.330411,"""PromethION""",0,16653212647
…,…,…,…,…,…,…
"""ERR15935028""","""s__JAFBIX01 sp021531895""",137.764068,0.992922,"""PromethION""",1,20643064873
"""ERR15934940""","""s__Bariatricus sp004560705""",108.242687,0.99321,"""PromethION""",1,31254201876
"""ERR15934725""","""s__Bariatricus sp004560705""",102.132863,0.993625,"""PromethION""",1,23038537820


In [14]:
by_species = sum_df.group_by('species').agg(
    min_breadth=pl.col('breadth').min().round(3),
    med_breadth=pl.col('breadth').median().round(3),
    max_breadth=pl.col('breadth').max().round(3),
    min_depth=pl.col('depth_all').min().round(2),
    med_depth=pl.col('depth_all').median().round(1),
    max_depth=pl.col('depth_all').max().round(1),
)

In [45]:
by_species.sort('min_depth').style

species,min_breadth,med_breadth,max_breadth,min_depth,med_depth,max_depth
s__Ornithospirochaeta sp022785155,0.057,0.891,0.978,0.2,5.0,14.1
s__Cryptobacteroides sp900546925,0.3,0.977,0.986,0.5,10.5,67.9
s__Cryptobacteroides sp000434935,0.35,0.83,0.887,0.56,6.1,30.9
s__Cryptobacteroides sp000432655,0.39,0.894,0.935,0.64,6.5,18.3
s__Cryptobacteroides sp034089285,0.527,0.981,0.991,1.13,16.1,61.5
s__UBA2868 sp004552595,0.192,0.7,0.929,1.81,18.0,52.3
s__Prevotella sp002251295,0.421,0.957,0.98,1.89,21.4,330.3
s__Colivicinus sp002299675,0.335,0.977,0.986,2.09,13.5,47.2
s__Sodaliphilus sp004557565,0.598,0.975,0.983,2.3,58.5,311.4
s__Fimisoma sp002320005,0.341,0.928,0.953,2.91,20.2,78.0


In [51]:
sum_df['breadth'].describe()

statistic,value
str,f64
"""count""",512.0
"""null_count""",0.0
"""mean""",0.883145
"""std""",0.153136
"""min""",0.056841
"""25%""",0.861999
"""50%""",0.943947
"""75%""",0.975082
"""max""",0.995247


In [52]:
len(sum_df)

512

In [56]:
len(sum_df.filter(pl.col('breadth') >= .5)) / 512

0.94921875

In [57]:
sum_df['depth_all'].describe()

statistic,value
str,f64
"""count""",512.0
"""null_count""",0.0
"""mean""",130.345946
"""std""",478.414332
"""min""",0.204572
"""25%""",8.784293
"""50%""",20.429855
"""75%""",48.453538
"""max""",5257.163197


In [58]:
len(sum_df.filter(pl.col('depth_all') >= 1)) / 512

0.990234375

In [50]:
(sum_df.sort('breadth')
 .select(['metag', 'species', 'depth_all', 'breadth'])
 .style.fmt_number(columns=["depth_all", "breadth"], decimals=2)
)

metag,species,depth_all,breadth
ERR15934505,s__Ornithospirochaeta sp022785155,0.20,0.06
ERR15934505,s__UBA2868 sp004552595,2.94,0.19
ERR15934505,s__Cryptobacteroides sp900546925,0.50,0.30
ERR15939764,s__UBA2868 sp004552595,1.81,0.33
ERR15934783,s__UBA2868 sp004552595,18.44,0.33
ERR15934927,s__UBA2868 sp004552595,43.89,0.33
ERR15934513,s__UBA2868 sp004552595,51.65,0.33
ERR15939764,s__Colivicinus sp002299675,2.26,0.34
ERR15934505,s__Fimisoma sp002320005,2.91,0.34
ERR15934764,s__UBA2868 sp004552595,19.39,0.35


In [44]:
by_species_healthy = (
    sum_df
    .filter(pl.col('is_diarrhea') == 0)
    .group_by('species').agg(
        min_breadth=pl.col('breadth').min().round(3),
        med_breadth=pl.col('breadth').median().round(3),
        max_breadth=pl.col('breadth').max().round(3),
        min_depth=pl.col('depth_all').min().round(2),
        med_depth=pl.col('depth_all').median().round(1),
        max_depth=pl.col('depth_all').max().round(1),
    )
)
by_species_healthy.sort('min_depth').style

species,min_breadth,med_breadth,max_breadth,min_depth,med_depth,max_depth
s__Ornithospirochaeta sp022785155,0.645,0.938,0.977,4.66,7.4,14.1
s__Colivicinus sp002299675,0.964,0.969,0.983,8.62,13.9,47.2
s__Cryptobacteroides sp000432655,0.906,0.924,0.927,10.74,17.4,18.3
s__Fimisoma sp002320005,0.731,0.927,0.953,11.3,31.9,51.6
s__Floccifex porci,0.45,0.59,0.923,11.98,20.2,40.5
s__Mogibacterium_A kristiansenii,0.62,0.922,0.968,12.32,26.7,48.7
s__Cryptobacteroides sp000434935,0.845,0.866,0.887,12.33,20.9,30.9
s__UBA2868 sp004552595,0.33,0.34,0.566,18.44,36.2,52.3
s__Cryptobacteroides sp900546925,0.978,0.984,0.986,33.43,44.6,67.9
s__Cryptobacteroides sp034089285,0.986,0.99,0.991,34.34,47.5,61.5


In [19]:
by_species_diarrhea = (
    sum_df
    .filter(pl.col('is_diarrhea') == 1)
    .group_by('species').agg(
        min_breadth=pl.col('breadth').min().round(3),
        med_breadth=pl.col('breadth').median().round(3),
        max_breadth=pl.col('breadth').max().round(3),
        min_depth=pl.col('depth_all').min().round(2),
        med_depth=pl.col('depth_all').median().round(1),
        max_depth=pl.col('depth_all').max().round(1),
    )
)
by_species_diarrhea.sort('min_depth').style

species,min_breadth,med_breadth,max_breadth,min_depth,med_depth,max_depth
s__Ornithospirochaeta sp022785155,0.057,0.864,0.978,0.2,3.8,10.4
s__Cryptobacteroides sp900546925,0.3,0.97,0.985,0.5,9.4,19.6
s__Cryptobacteroides sp000434935,0.35,0.819,0.886,0.56,5.3,12.6
s__Cryptobacteroides sp000432655,0.39,0.889,0.935,0.64,5.9,16.6
s__Cryptobacteroides sp034089285,0.527,0.973,0.991,1.13,13.1,35.9
s__UBA2868 sp004552595,0.192,0.824,0.929,1.81,14.2,30.5
s__Prevotella sp002251295,0.421,0.95,0.977,1.89,17.4,44.9
s__Colivicinus sp002299675,0.335,0.978,0.986,2.09,13.5,43.9
s__Sodaliphilus sp004557565,0.598,0.975,0.983,2.3,46.2,127.1
s__Fimisoma sp002320005,0.341,0.93,0.953,2.91,18.6,78.0


In [20]:
combo_df.head(1)

metag,gene,len,hits,breadth,depth_all,depth_cov,species,acc,weighted_depth,weighted_breadth
str,str,i64,i64,f64,f64,f64,str,str,f64,f64
"""ERR15909852""","""FR882202.1""",72839,5938,0.081522,0.088785,1.089087,"""s__Cryptobacteroides sp0004349…","""GCA_000434935.1""",6467.0,5938.0


In [26]:
combo_df

metag,gene,len,hits,breadth,depth_all,depth_cov,species,acc,weighted_depth,weighted_breadth
str,str,i64,i64,f64,f64,f64,str,str,f64,f64
"""ERR15909852""","""FR882202.1""",72839,5938,0.081522,0.088785,1.089087,"""s__Cryptobacteroides sp0004349…","""GCA_000434935.1""",6467.0,5938.0
"""ERR15909852""","""SFNX01000131.1""",4530,0,0.0,0.0,NaN,"""s__UBA2868 sp004552595""","""GCA_004552595.1""",0.0,0.0
"""ERR15909852""","""NZ_JAFBIX010000025.1""",46446,8035,0.172997,0.416268,2.406223,"""s__JAFBIX01 sp021531895""","""GCF_021531895.1""",19334.0,8035.0
"""ERR15909852""","""JALFRH010000010.1""",9875,105,0.010633,0.031899,3.0,"""s__Ornithospirochaeta sp022785…","""GCA_022785155.1""",315.0,105.0
"""ERR15909852""","""JALFNN010000087.1""",63285,24763,0.391293,0.56511,1.444211,"""s__Bariatricus sp004560705""","""GCA_022774325.1""",35763.0,24763.0
…,…,…,…,…,…,…,…,…,…,…
"""ERR15942255""","""SFNX01000182.1""",3218,0,0.0,0.0,NaN,"""s__UBA2868 sp004552595""","""GCA_004552595.1""",0.0,0.0
"""ERR15942255""","""JALFNN010000085.1""",4012,0,0.0,0.0,NaN,"""s__Bariatricus sp004560705""","""GCA_022774325.1""",0.0,0.0
"""ERR15942255""","""SFNX01000139.1""",3818,0,0.0,0.0,NaN,"""s__UBA2868 sp004552595""","""GCA_004552595.1""",0.0,0.0


In [35]:
xx = []

for species in combo_df['species'].unique():
    xx_df = combo_df.filter(pl.col('species') == species)
    xx_df = xx_df.join(prjeb_df, left_on='metag', right_on='Run', how='inner')
    healthy_df = xx_df.filter(pl.col('is_diarrhea') == 0)
    diarrhea_df = xx_df.filter(pl.col('is_diarrhea') == 1)

    healthy_mean = healthy_df['depth_all'].mean()
    healthy_std = healthy_df['depth_all'].std()

    diarrhea_mean = diarrhea_df['depth_all'].mean()
    diarrhea_std = diarrhea_df['depth_all'].std()

    print(f"{healthy_mean:.1f} +/- {healthy_std:.1f} vs {diarrhea_mean:.1f} +/- {diarrhea_std:.1f} - {species}")
    

46.2 +/- 17.2 vs 9.3 +/- 5.3 - s__Cryptobacteroides sp900546925
286.5 +/- 1066.0 vs 152.3 +/- 525.8 - s__JAFBIX01 sp021531895
613.5 +/- 2403.5 vs 79.3 +/- 297.2 - s__Prevotella sp002251295
16.6 +/- 11.4 vs 5.9 +/- 4.4 - s__Cryptobacteroides sp000432655
233.9 +/- 354.2 vs 69.3 +/- 208.7 - s__Sodaliphilus sp004557565
47.9 +/- 20.4 vs 14.3 +/- 10.6 - s__Cryptobacteroides sp034089285
45.8 +/- 184.2 vs 57.5 +/- 215.1 - s__Floccifex porci
21.6 +/- 22.0 vs 17.1 +/- 21.3 - s__Colivicinus sp002299675
34.3 +/- 81.7 vs 24.3 +/- 64.2 - s__Fimisoma sp002320005
1122.2 +/- 1322.4 vs 1605.2 +/- 1260.2 - s__Lactobacillus amylovorus
7.4 +/- 14.8 vs 4.1 +/- 5.7 - s__Ornithospirochaeta sp022785155
83.5 +/- 362.3 vs 84.8 +/- 393.2 - s__Mogibacterium_A kristiansenii
46.7 +/- 146.4 vs 19.4 +/- 114.5 - s__UBA2868 sp004552595
405.5 +/- 824.1 vs 35.2 +/- 105.4 - s__Prevotella sp000434975
20.8 +/- 24.8 vs 5.8 +/- 8.6 - s__Cryptobacteroides sp000434935
90.2 +/- 116.5 vs 53.1 +/- 63.4 - s__Bariatricus sp004560705
